# Прототипирование

Интерактивная среда превращает пользователей и заказчиков из пассивных наблюдателей в соавторов системы: наглядное представление текущих результатов и принципов работы системы и модели позволяет оперативно получать обратную связь, что на ранних стадиях проекта будет учтено в доработках, и которые войдут в финальную версию как системы дашборда, так и самой модели машинного обучения.

С практической точки зрения польза прототипирования заключается в реализации итеративного подхода к разработке, что минимизирует риски создания оторванного от реальности продукта, который может привести сначала к приостановке проекта, а потом и к полному закрытию.

Разработка дашбордов сегодня вышла за рамки сложных BI-систем, требующих работы целых отделов. Современные инструменты позволяют аналитикам и разработчикам превращать данные в интерактивные веб-приложения, не погружаясь в тонкости фронтенд-технологий.

Одним из наиболее популярных решений для быстрого прототипирования является фреймворк с открытым исходным кодом Streamlit, предназначенный для быстрой трансформации аналитических скриптов на языке Python в интерактивные веб-приложения.

1.	Для проекта создания дашбордов лучше сделать отдельное виртуальное окружение командой python -m venv venv.
2.	Активировать его командой source venv/Scripts/activate для Windows или source venv/bin/activate для macOS/Linux.
3.	Установить библиотеку командой pip install streamlit==1.56.0, где лучше указать конкретную версию.


# Запустим прототип

Код лучше сразу писать с помощью IDE в файле .py (например, start.py) и запускать из терминала командой streamlit run start.py. Работа из Jupyter Notebook потребует использования магической команды %%writefile start.py.
Запущенный скрипт автоматически перебросит тебя на страницу в браузере по адресу: http://localhost:8501, где уже будет доступен подготовленный макет.

Простой пример:

In [ ]:
import streamlit as st

st.title("Система готова к работе")
st.write("Это минимальное приложение для проверки интерфейса.")

if st.button("Нажми меня"):
    st.balloons()  # Визуальный эффект для подтверждения интерактивности

ModuleNotFoundError: No module named 'streamlit'

# Навигация

Зададим название страницы page_title="Панель управления" и укажем через параметр layout="wide" отображение приложения на всю ширину экрана браузера.

In [ ]:
st.set_page_config(page_title="Панель управления", layout="wide")

Разделить страницу на несколько вертикальных контейнеров можно через функцию st.columns. Она принимает либо целое число (равные доли), либо список чисел (пропорциональное соотношение).

In [ ]:
col1, col2, col3, col4 = st.columns([1, 1, 1, 4])

with col1:
    st.button("🏠 Главная", key="btn_home", use_container_width=True)

with col2:
    st.button("📈 Прогноз увольнения", key="btn_analytics", use_container_width=True)

with col3:
    st.button("⚙️ Настройки", key="btn_settings", use_container_width=True)

# Главная страница

In [ ]:
st.title("Панель управления")

col1, col2, col3 = st.columns(3)
with col1:
    st.markdown("### 👥 Динамика кадров")
    st.write("Общие сведения и движение сотрудников")
    st.button("Открыть отчет", key="btn1")

    st.markdown("### 🎓 О школе")
    st.write("Общая информация о школе")
    st.button("В разработке...", disabled=True, key="btn4")

with col2:
    st.markdown("### 📚 Обучение")
    st.write("Повышение квалификации и переподготовка сотрудников")
    st.button("В разработке...", disabled=True, key="btn2")

with col3:
    st.markdown("### ⏱️ Нагрузка")
    st.write("Количество уроков и часов у сотрудника")
    st.button("В разработке...", disabled=True, key="btn3")

Обрабатывать все страницы дашборда в одном файле сложно, и это приведёт к проблемам поддержки позже.
Поэтому отрисовка содержания страниц может быть выполнена с помощью создания отдельных файлов Python с содержанием в отдельной папке, например pages_content, которые будут импортированы в главный скрипт.

In [ ]:
# ./pages_content/home.py
import streamlit as st

def show():
    st.title("Панель управления")

    col1, col2, col3 = st.columns(3)
    with col1:
        st.markdown("### 👥 Динамика кадров")
        st.write("Общие сведения и движение сотрудников")
        # st.button("Открыть отчет", key="btn1")
        if st.button("Открыть отчет", key="btn1"):
            st.switch_page("./pages/report.py") # Переход на страницу в папке pages

        st.markdown("### 🎓 О школе")
        st.write("Общая информация о школе")
        st.button("В разработке...", disabled=True, key="btn4")

    with col2:
        st.markdown("### 📚 Обучение")
        st.write("Повышение квалификации и переподготовка сотрудников")
        st.button("В разработке...", disabled=True, key="btn2")

    with col3:
        st.markdown("### ⏱️ Нагрузка")
        st.write("Количество уроков и часов у сотрудника")
        st.button("В разработке...", disabled=True, key="btn3")

Импорт в главном скрипте можно выполнить так:

In [ ]:
from pages_content import home

# Состояние

Для реализации приложения из нескольких страниц нам нужно задать «состояние» приложения через st.session_state, что позволяет сохранять данные при перезапуске скрипта, обмениваться информацией между разными частями интерфейса и создавать логику переключения экранов. Например, если у нас будет несколько страниц, то при первой загрузке страницы необходимо указать загрузку «Главной» страницы.

In [ ]:
# Инициализация состояния
if 'current_page' not in st.session_state:
    st.session_state.current_page = "Главная"

In [ ]:
# Проверка кликов до отрисовки
if st.session_state.get("btn_home"):
    st.session_state.current_page = "Главная"
if st.session_state.get("btn_analytics"):
    st.session_state.current_page = "Аналитика"
if st.session_state.get("btn_settings"):
    st.session_state.current_page = "Настройки"

In [ ]:
# Отрисовка контента
if st.session_state.current_page == "Главная":
    home.show()

# Дашборд

В современной архитектуре Streamlit также реализована нативная поддержка многостраничности. Для активации многостраничного режима необходимо строго соблюдать иерархию файлов. Основной файл приложения находится в корне, а все дополнительные страницы — в специальной папке с фиксированным именем pages/. Streamlit автоматически сканирует содержимое папки pages/ и формирует навигационное меню в боковой панели. Каждый файл внутри папки pages/ рассматривается как независимый скрипт.

In [ ]:
# ./pages/report.py
import streamlit as st
import pandas as pd
import numpy as np

# Настройка страницы
st.set_page_config(page_title="Отчет", layout="wide")

st.title("📊 Динамика кадров")

# Показатели
st.subheader("Ключевые показатели")
col1, col2, col3, col4 = st.columns(4)

with col1:
    st.metric(label="Динамика сотрудников", value="2,456", delta="12.3%")
with col2:
    st.metric(label="В зоне риска", value="1,847", delta="-3.7%")
with col3:
    st.metric(label="Средний возраст", value="45", delta="2.1%")
with col4:
    st.metric(label="Проходят ПК за период", value="73.2%", delta="0.5%")

st.markdown("---")

Работа с данными для последующей визуализации сопряжена с важной особенностью Streamlit относительно повторного выполнения скрипта после любого изменения. Чтобы команда загрузки файлов не выполнялась каждый раз заново, необходимо использовать декоратор @st.cache_data.

In [ ]:
@st.cache_data
def load_data(path):
    # Здесь может быть сложная логика очистки данных после открытия
    data = pd.read_excel(path)

    # cols_to_drop = ['Предмет', 'Параллель', 'Имеет классное руководство']
    # data = data.drop(columns=cols_to_drop, errors='ignore')
    # data = data.drop_duplicates(subset=['id'], keep='first')

    return data

df = load_data(data_path)

Отдельно нужно помнить про принципы работы с путями в Python и использовать библиотеку pathlib для динамического определения корня проекта.

In [ ]:
from pathlib import Path

base_path = Path(__file__).parent.parent # Поднимаемся на уровень выше из папки pages
data_path = base_path / "data" / "dataset.csv"

# Графики

In [ ]:
# Распределение по возрасту
age_dist = df['Возраст'].value_counts().sort_index()
st.bar_chart(age_dist)

In [ ]:
# Распределение по полу
gender_counts = df['Пол'].value_counts()
st.bar_chart(gender_counts)

In [ ]:
# Представление датафрейма
st.dataframe(df, use_container_width=True)

In [ ]:
# Добавление вкладок
tab1, tab2 = st.tabs(["Базовые", "Расширенные"])
with tab1:
    pass
with tab2:
    pass

Хорошим вариантом построения интерактивных графиков является использование библиотеки Plotly. Она позволяет существенно расширить доступный набор графиков.

In [ ]:
import plotly.express as px

In [ ]:
# Круговая диаграмма по полу
gender_counts = df['Пол'].value_counts()

fig = px.pie(
    values=gender_counts.values,
    names=gender_counts.index,
    hole=0.5
)

st.plotly_chart(fig, use_container_width=True)

In [ ]:
# Ящик с усами
fig_box = px.box(
    df,
    x="Пол",
    y="Возраст",
    points="outliers",
    title="Анализ выбросов по возрасту"
)
st.plotly_chart(fig_box, use_container_width=True)

# Фильтры

В архитектуре Streamlit есть встроенные блок Sidebar (боковая панель), который представляет собой выделенную область интерфейса, предназначенную для размещения элементов управления, которые должны оставаться доступными независимо от контента в центральной части экрана. С точки зрения методологии UX/UI для аналитических систем сайдбар реализует принцип разделения контекстов: «управление» выносится влево, а «результат» (графики и выводы) занимает основное пространство.

По умолчанию Streamlit обновляет всё приложение сразу при каждом изменении виджета (слайдера или списка). Чтобы этого избежать и запускать расчёт только тогда, когда мы настроили все параметры, используют форму (st.form).

In [ ]:
with st.sidebar:
    st.header("Настройка фильтров")

    with st.form("my_filter_form"):
        gender = st.multiselect("Выберите пол", options=df['Пол'].unique(), default=df['Пол'].unique())
        age = st.slider("Возраст", 16, 100, (20, 50))

        submit_button = st.form_submit_button(label='Применить фильтры')

Логика срабатывает только при нажатии кнопки или при первой загрузке.

In [ ]:
if submit_button:
    filtered_df = df[(df['Пол'].isin(gender)) & (df['Возраст'].between(age[0], age[1]))]
    st.success(f"Найдено записей: {len(filtered_df)}")
    st.dataframe(filtered_df)